# Data Ingestion — step by step

This notebook runs the ingestion pipeline one stage at a time so you can see the
data change shape at each step.

**What ingestion does:** read a folder of documentation files and turn each one into
a `Document` with four required metadata fields — `source_file`, `page_id`,
`sdk_version`, `page_type`. If a file can't be read, record *why* and keep going.

```
file  →  1. pick extractor  →  2. extract text  →  3. resolve metadata  →  4. check identity  →  Document
              (by extension)      (bytes → text)      (fill the gaps)        (unique?)
```

Chunking, embedding and retrieval are **later phases** and are not in this code yet.

## Setup

Run from the repo root so relative paths like `docs` resolve, and so `import app`
works. This cell is safe to run from either the repo root or `notebooks/`.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("working directory:", ROOT)
print("app package found :", (ROOT / "app" / "rag").is_dir())

working directory: C:\Users\HariPrasathSelvam\Desktop\Gen-AI-Dev
app package found : True


## The two corpora

- **`docs/`** — clean. Every file has front matter. Everything loads.
- **`sample-corpus/`** — deliberately broken. This is the interesting one: it proves
  a bad file gets quarantined instead of killing the run.

In [2]:
for corpus in ("docs", "sample-corpus"):
    print(f"--- {corpus}/ ---")
    for path in sorted(Path(corpus).rglob("*")):
        if path.is_file():
            rel = path.relative_to(corpus).as_posix()
            print(f"  {rel:<44} {path.stat().st_size:>6} B")
    print()

--- docs/ ---
  v2/client-send.md                               670 B
  v3/client-send.md                               972 B
  v3/client-stream.md                             694 B

--- sample-corpus/ ---
  misc/bad-page-type.md                           289 B
  misc/corrupt.pdf                                 25 B
  misc/diagram.png                                 40 B
  misc/empty.md                                     0 B
  v2/guide/migration.txt                          244 B
  v3/reference/client-batch.pdf                  1300 B
  v3/reference/client-close.docx                36859 B
  v3/reference/client-send.md                     413 B
  v3/reference/client-stream.html                 493 B
  v3/reference/zz-duplicate-of-client-send.md     310 B



## Stage 1 — pick an extractor

`app/rag/extractors/__init__.py` is just a dictionary mapping file extension to a
handler. Adding a new format means writing one module and adding one line.

If nothing matches, the answer is `None` — which means **"record it as skipped"**,
never "silently pretend the file isn't there".

In [3]:
from app.rag.extractors import SUPPORTED_EXTENSIONS, get_extractor

print("supported:", ", ".join(SUPPORTED_EXTENSIONS), "\n")

for name in ("guide.md", "page.html", "manual.pdf", "notes.docx", "diagram.png"):
    extractor = get_extractor(Path(name))
    label = type(extractor).__name__ if extractor else "None  ->  SKIPPED"
    print(f"  {name:<14} {label}")

supported: .docx, .htm, .html, .markdown, .md, .pdf, .rst, .txt 

  guide.md       MarkdownExtractor
  page.html      HtmlExtractor
  manual.pdf     PdfExtractor
  notes.docx     DocxExtractor
  diagram.png    None  ->  SKIPPED


## Stage 2 — extract

Every extractor returns the same shape, an `Extracted`:

| field | meaning |
|---|---|
| `text` | the content |
| `front_matter` | what the file *declared* about itself (markdown can; a PDF can't) |
| `title_hint` | a title found *inside* the content — weaker than a declaration |

The file below has **no front matter at all**. Watch `front_matter` come back empty.

In [4]:
CORPUS = Path("sample-corpus")
target = CORPUS / "v3/reference/client-send.md"

extractor = get_extractor(target)
extracted = extractor.extract(target)

print("extractor    :", extractor.name)
print("front_matter :", extracted.front_matter, "  <- nothing declared")
print("title_hint   :", repr(extracted.title_hint), "  <- found the '# heading'")
print("\ntext:\n")
print(extracted.text.strip())

extractor    : markdown
front_matter : {}   <- nothing declared
title_hint   : 'Client.send()'   <- found the '# heading'

text:

# Client.send()

No front matter at all. Everything below must be derived: `sdk_version` from
the `v3` folder, `page_type` from the `reference` folder, `page_id` from the
filename, and the title from this heading.

| name             | type | default | required |
|------------------|------|---------|----------|
| payload          | dict | —       | yes      |
| retry_backoff_ms | int  | 250     | no       |


## Stage 3 — resolve metadata

This is the part that makes ingestion work on real corpora. Front matter is
**optional**; each field walks a chain until something answers:

```
sdk_version:  front matter  →  a folder named "v3"         →  "unknown"
page_type:    front matter  →  a folder named "reference"  →  "reference"
page_id:      front matter  →  the filename
title:        front matter  →  the "# heading"             →  the filename
```

`resolve()` also returns **which link answered**. That matters: "the author declared
it" and "I guessed it from a folder name" are different levels of trust, and only one
is worth investigating when a value looks wrong.

In [5]:
from app.rag.metadata import resolve

relative = target.relative_to(CORPUS)
values, sources = resolve(relative, extracted)

print(f"path: {relative.as_posix()}\n")
print(f"{'field':<13} {'value':<18} came from")
print("-" * 48)
for field in ("sdk_version", "page_type", "page_id", "title"):
    value = values[field]
    value = value.value if hasattr(value, "value") else value
    print(f"{field:<13} {str(value):<18} {sources[field].value}")

path: v3/reference/client-send.md

field         value              came from
------------------------------------------------
sdk_version   v3                 path
page_type     reference          path
page_id       client-send        filename
title         Client.send()      content


### Front matter always wins

Same resolver, but now the file declares its own values — so the folder name is ignored.

In [6]:
from app.rag.extractors import Extracted

declared = Extracted(text="body", front_matter={"sdk_version": "v2", "page_type": "guide"})
values2, sources2 = resolve(Path("v3/reference/thing.md"), declared)

print("file sits in  : v3/reference/")
print("but declares  : sdk_version=v2, page_type=guide\n")
print("sdk_version ->", values2["sdk_version"], "from", sources2["sdk_version"].value)
print("page_type   ->", values2["page_type"].value, "from", sources2["page_type"].value)

file sits in  : v3/reference/
but declares  : sdk_version=v2, page_type=guide

sdk_version -> v2 from front_matter
page_type   -> guide from front_matter


## Stage 4 — the whole corpus

`load_documents()` is the conductor: it loops over every file, runs stages 1–3,
catches errors per file, and returns an `IngestReport`.

**The key idea: per-file problems are recorded, not raised.** One malformed page in a
corpus of fifty thousand must not stop the other 49,999. Only a missing docs root is
fatal.

In [7]:
from app.rag.loader import load_documents

report = load_documents(CORPUS)
print(report.counts())

{'loaded': 5, 'failed': 4, 'skipped': 1}


In [8]:
print(f"{'sdk':<6} {'page_id':<16} {'type':<10} {'format':<9} {'chars':>6}  source_file")
print("-" * 78)
for doc in sorted(report.documents, key=lambda d: (d.sdk_version, d.page_id)):
    print(
        f"{doc.sdk_version:<6} {doc.page_id:<16} {doc.page_type.value:<10} "
        f"{doc.source_format:<9} {len(doc.text):>6}  {doc.source_file}"
    )

sdk    page_id          type       format     chars  source_file
------------------------------------------------------------------------------
v2     migration        guide      text         244  v2/guide/migration.txt
v3     client-batch     reference  pdf          182  v3/reference/client-batch.pdf
v3     client-close     reference  docx         180  v3/reference/client-close.docx
v3     client-send      reference  markdown     411  v3/reference/client-send.md
v3     client-stream    reference  html         149  v3/reference/client-stream.html


### The quarantine

Four files failed, each at a different point. `stage` tells you where it broke:

- **extract** — couldn't turn the bytes into text
- **metadata** — a *declared* value was invalid (so it's an error, not something to guess around)
- **validate** — the document was fine, but its identity collided with one already loaded

In [9]:
for failure in report.failures:
    print(f"[{failure.stage.value:<8}] {failure.source_file}")
    print(f"           {failure.reason}\n")

print("SKIPPED (never even read):")
for skipped in report.skipped:
    print(f"  {skipped.source_file} — {skipped.reason}")

[metadata] misc/bad-page-type.md
           page_type 'tutorial' must be one of reference, guide, changelog

[extract ] misc/corrupt.pdf
           unreadable PDF: Stream has ended unexpectedly

[extract ] misc/empty.md
           no text content

[validate] v3/reference/zz-duplicate-of-client-send.md
           duplicate page v3/client-send — already ingested from v3/reference/client-send.md

SKIPPED (never even read):
  misc/diagram.png — unsupported file type .png


### Duplicate identity

`(sdk_version, page_id)` is a page's identity and must be unique — otherwise one
document would silently overwrite another later in the index. The **first** file wins;
the second is quarantined and the message names both.

In [10]:
from app.rag.models import Stage

for failure in report.failures:
    if failure.stage is Stage.validate:
        print("quarantined:", failure.source_file)
        print("reason     :", failure.reason)

quarantined: v3/reference/zz-duplicate-of-client-send.md
reason     : duplicate page v3/client-send — already ingested from v3/reference/client-send.md


## Why PDF is a bad source for reference docs

The same parameter table, ingested from two formats. **PDF stores glyph positions,
not structure** — so the row falls apart and nothing downstream can rebuild it.
DOCX keeps cells as real objects, so rows survive.

This matters for the *next* phase: chunking can only preserve structure that
ingestion managed to keep.

In [11]:
by_format = {d.source_format: d for d in report.documents}

for fmt in ("pdf", "docx"):
    print(f"===== from {fmt.upper()} =====")
    print(by_format[fmt].text)
    print()

===== from PDF =====
Client.send_batch()
Sends many requests, amortising connection setup across the batch.
name type default required
payloads list - yes
concurrency int 8 no
retry_backoff_ms int 250 no

===== from DOCX =====
Client.close()

Releases every pooled connection and stops the reaper thread.

| name | type | default | required |

| timeout_ms | int | 5000 | no |

| force | bool | False | no |



## The clean corpus, and the version filter

`docs/` has front matter everywhere, so nothing is quarantined — `report.ok` is True.

The `sdk_version` filter is what keeps a run scoped to *new* pages instead of
re-reading the whole documentation tree.

In [12]:
clean = load_documents(Path("docs"))
print("all versions:", clean.counts(), "  ok =", clean.ok)
for doc in clean.documents:
    print("   ", doc.sdk_version, doc.page_id)

only_v3 = load_documents(Path("docs"), sdk_version="v3")
print("\nv3 only    :", only_v3.counts())
for doc in only_v3.documents:
    print("   ", doc.sdk_version, doc.page_id)

all versions: {'loaded': 3, 'failed': 0, 'skipped': 0}   ok = True
    v2 client-send
    v3 client-send
    v3 client-stream

v3 only    : {'loaded': 2, 'failed': 0, 'skipped': 0}
    v3 client-send
    v3 client-stream


## One document in full

This is what the rest of the RAG pipeline will eventually receive.

In [13]:
doc = clean.documents[0]

print("required metadata:")
for key, value in doc.metadata().items():
    print(f"  {key:<13} {value}")

print("\nhow each field was resolved:")
for key, source in doc.metadata_sources.items():
    print(f"  {key:<13} {source.value}")

print(f"\ntext ({len(doc.text)} chars):\n")
print(doc.text.strip()[:400])

required metadata:
  source_file   v2/client-send.md
  page_id       client-send
  sdk_version   v2
  page_type     reference

how each field was resolved:
  sdk_version   front_matter
  page_type     front_matter
  page_id       front_matter
  title         front_matter

text (581 chars):

# Client.send()

Sends a single request to the configured endpoint and blocks until a response
is received. Retries are not performed automatically in v2; callers implement
their own retry loop around the call.

## Parameters

| name             | type | default | required |
|------------------|------|---------|----------|
| payload          | dict | —       | yes      |
| timeout_ms       | int  


## Same thing from the command line

```bash
python -m app.rag.ingest                        # clean corpus
python -m app.rag.ingest --docs sample-corpus   # messy corpus, exits 0
python -m app.rag.ingest --docs sample-corpus --strict   # same corpus, exits 1
python -m app.rag.ingest --version v3           # one SDK version
python -m app.rag.ingest --show                 # print every document body
python -m app.rag.ingest --json                 # machine-readable report
```

And over HTTP (`uvicorn app.main:app --reload`):

```
POST /api/v1/ingest/run          {"docs_root": "sample-corpus"}
GET  /api/v1/ingest/report
GET  /api/v1/ingest/documents?sdk_version=v3
GET  /api/v1/ingest/documents/{sdk_version}/{page_id}
```

---

## Where to read the code

Start with **`app/rag/loader.py`** — the `for path in sorted(...)` loop in
`load_documents` *is* the program, about 40 lines. Everything else is a helper it
calls.

| file | job |
|---|---|
| `app/rag/models.py` | the shapes: `Document`, `IngestReport`, `FailedFile`, `SkippedFile` |
| `app/rag/extractors/` | stage 1–2, one module per format |
| `app/rag/metadata.py` | stage 3, the fallback chains |
| `app/rag/loader.py` | the conductor — loops, catches, builds the report |
| `app/rag/ingest.py` | CLI |
| `app/rag/service.py` | keeps the last report in memory for the API |
| `app/api/routes/ingest.py` | HTTP wrapper |

**Next phase: chunking** — splitting these documents into retrievable pieces without
cutting a parameter row away from its header, or a code fence in half.